# LLaVA-Med — Deletion Faithfulness Experiment

Clean per-token deletion experiment for `microsoft/llava-med-v1.5-mistral-7b`.

In [ ]:
# Standard install (local / non-Colab)
%pip install --quiet \
    "transformers==4.36.2" \
    "tokenizers==0.15.2" \
    "accelerate==0.21.0" \
    "bitsandbytes>=0.41.0" \
    "datasets" \
    "pillow" \
    "numpy" "pandas" "tqdm" "scipy" \
    "huggingface_hub"

# --- COLAB ONLY: uncomment the block below when running in Google Colab ---
# # Clean conflicting core binaries first
# %pip uninstall -y numpy scipy pandas scikit-learn transformers tokenizers accelerate bitsandbytes llava-med || true
# # Install ABI-compatible scientific stack
# %pip install "numpy==1.26.4" "scipy==1.11.4" "pandas==2.1.4" "scikit-learn==1.2.2"
# # Install LLaVA-Med compatible model stack
# %pip install "transformers==4.36.2" "tokenizers==0.15.2" "accelerate==0.21.0" "bitsandbytes==0.41.0"
# # Install official LLaVA-Med package
# %pip install "git+https://github.com/microsoft/LLaVA-Med.git"
# print("Installed pinned ABI-compatible stack. Restart runtime, then run from Cell 2.")

print("Installation complete.")

In [ ]:
import os
from huggingface_hub import login

# --- COLAB ONLY: uncomment to mount Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')

# Set HF_TOKEN in your shell before launching: export HF_TOKEN=hf_...
# In Colab: use the Secrets panel (key icon) or paste your token below.
token = os.environ.get('HF_TOKEN')
if token:
    login(token=token)
else:
    login()  # will prompt interactively

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['PYTORCH_NVFUSER_DISABLE'] = '1'
os.environ['TORCH_NVFUSER_DISABLE'] = '1'
os.environ['PYTORCH_JIT_USE_NNC_NOT_NVFUSER'] = '1'
print('[env] CUDA_LAUNCH_BLOCKING=1, NVFuser disabled')

In [ ]:
import sys
from pathlib import Path

# Assumes the notebook lives inside the llava_family/ directory.
# If running from a different working directory, set MODULE_DIR explicitly.
MODULE_DIR = Path.cwd()
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
print(f'Module path: {MODULE_DIR}')

# --- COLAB ONLY: uncomment and update BASE_DIR to your Drive path ---
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)
# BASE_DIR = Path("/content/drive/Othercomputers/My Mac/Thesis/llava_med")
# sys.path.insert(0, str(BASE_DIR))
# print(f'Base dir: {BASE_DIR}')

In [ ]:
from config import Config
from pathlib import Path

# ── Dataset toggle ────────────────────────────────────────────────────────
USE_COCO = False  # True -> COCO (lmms-lab/COCO-Caption val) | False -> ROCO v2

cfg = Config()
cfg.model_id            = 'microsoft/llava-med-v1.5-mistral-7b'
cfg.load_in_4bit        = True
cfg.attn_implementation = 'eager'

if USE_COCO:
    cfg.dataset_name   = 'lmms-lab/COCO-Caption'
    cfg.dataset_split  = 'val'
    cfg.image_column   = 'image'
    cfg.caption_column = 'answer'
    cfg.prompt         = 'Generate a caption for this image.'
    cfg.output_dir     = 'results_deletion_coco'
else:
    cfg.dataset_name   = 'eltorio/ROCOv2-radiology'
    cfg.dataset_split  = 'train'
    cfg.image_column   = 'image'
    cfg.caption_column = 'caption'
    cfg.prompt         = 'Write a single-sentence radiology caption for this medical image.'
    cfg.output_dir     = 'results_deletion_roco'

cfg.num_samples              = 500
cfg.max_new_tokens           = 100
cfg.attention_layer_strategy = 'global'
cfg.methods                  = ['attention', 'gmar_l2', 'gmar_l1', 'gradcam']
cfg.mask_ratios              = [0.1, 0.2, 0.3, 0.4, 0.5]
cfg.save_visualizations      = False
cfg.enable_sink_suppression        = False
cfg.sink_consistency_threshold     = 0.75
cfg.sink_floor_percentile          = 10.0
cfg.rollout_block_generated_tokens = False
# LLaVA-Med v1.5 (mistral-7b): CLIP ViT-L/14 @ 336px -> 24x24 = 576 image tokens
# Original LLaVA-Med v1.0 (224px): set grid=16 for 16x16 = 256 tokens
cfg.image_token_grid  = 24
cfg.image_token_index = 32000

CONTENT_ONLY = True

print(f'[config] Model   : {cfg.model_id}')
print(f'[config] Dataset : {"COCO" if USE_COCO else "ROCO v2"}')
print(f'[config] Output  : {cfg.output_dir}')
print(f'[config] Methods : {cfg.methods}')

In [ ]:
# --- COLAB ONLY: bitsandbytes CUDA 12.8 mismatch workaround ---
# Uncomment this entire cell if you hit bitsandbytes CUDA binary errors in Colab.
# This disables 4-bit quantisation and removes the incompatible bnb build.
#
# import os, sys, subprocess, importlib.util
# cfg.load_in_4bit = False
# os.environ['BITSANDBYTES_NOWELCOME'] = '1'
# bnb_installed = importlib.util.find_spec('bitsandbytes') is not None
# if bnb_installed:
#     print('[env] bitsandbytes detected; uninstalling for this runtime...')
#     subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'bitsandbytes'], check=False)
#     print('[env] bitsandbytes removed. Continue to model load cell.')
# else:
#     print('[env] bitsandbytes not installed; continue to model load cell.')
print('[bnb-fix] Colab bnb fix cell skipped (not in Colab).')

In [ ]:
# --- COLAB ONLY: ABI self-heal (run once before model load in Colab) ---
# Uncomment this entire cell if you encounter numpy/scipy/transformers binary
# incompatibility errors ('numpy.dtype size changed', etc.) in Colab.
#
# import os, sys, signal, subprocess
# PINNED = [
#     'numpy==1.26.4', 'scipy==1.11.4', 'pandas==2.1.4', 'scikit-learn==1.2.2',
#     'transformers==4.36.2', 'tokenizers==0.15.2', 'accelerate==0.21.0',
# ]
# def _abi_broken():
#     try:
#         import numpy as np; import numpy.random.mtrand
#         import scipy; import pandas; import sklearn; import transformers
#         return False, 'ABI looks healthy.'
#     except Exception as e:
#         return ('numpy.dtype size changed' in str(e)) or ('binary incompatibility' in str(e).lower()), str(e)
# broken, msg = _abi_broken()
# print(f'[abi-check] broken={broken}')
# if broken:
#     print('[abi-fix] Reinstalling pinned stack...')
#     subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
#                     'numpy', 'scipy', 'pandas', 'scikit-learn',
#                     'transformers', 'tokenizers', 'accelerate'], check=False)
#     subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
#                     '--force-reinstall', *PINNED], check=True)
#     print('[abi-fix] Reinstall complete. Restarting runtime now...')
#     os.kill(os.getpid(), signal.SIGKILL)
# else:
#     print('[abi-check] OK, continue to model-load cell.')
print('[abi-fix] Colab ABI self-heal cell skipped (not in Colab).')

In [ ]:
# --- COLAB ONLY: NVRTC fix (run once before model load in Colab) ---
# Uncomment this entire cell if you encounter libnvrtc-builtins CUDA errors in Colab.
#
# import os, sys, site, glob, ctypes, shutil, subprocess
# from pathlib import Path
# print('[nvrtc-fix] Installing NVRTC runtime wheels...')
# subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-cache-dir', '--upgrade',
#                 'nvidia-cuda-nvrtc-cu12', 'nvidia-cuda-runtime-cu12',
#                 'nvidia-cuda-nvrtc-cu13'], check=False)
# lib_dirs = []
# for sp in [*site.getsitepackages(), site.getusersitepackages()]:
#     for rel in ('nvidia/cuda_nvrtc/lib', 'nvidia/cuda_runtime/lib'):
#         d = Path(sp) / rel
#         if d.exists():
#             lib_dirs.append(d)
# if lib_dirs:
#     old = os.environ.get('LD_LIBRARY_PATH', '')
#     os.environ['LD_LIBRARY_PATH'] = ':'.join([str(d) for d in lib_dirs] + ([old] if old else []))
#     print('[nvrtc-fix] Added to LD_LIBRARY_PATH:', lib_dirs)
# all_builtins = [p for d in lib_dirs for p in sorted(d.glob('libnvrtc-builtins.so*'))]
# target = next((p for p in all_builtins if p.name == 'libnvrtc-builtins.so.13.0'), None)
# if target is None and all_builtins:
#     pref = [p for p in all_builtins if '.so.13' in p.name] or \
#            [p for p in all_builtins if '.so.12' in p.name]
#     if pref:
#         src = pref[0]; dst = src.parent / 'libnvrtc-builtins.so.13.0'
#         if not dst.exists(): dst.symlink_to(src.name)
#         target = dst
# if target:
#     for sys_dir in (Path('/usr/local/lib'), Path('/usr/lib/x86_64-linux-gnu')):
#         try:
#             sys_dir.mkdir(parents=True, exist_ok=True)
#             dst = sys_dir / 'libnvrtc-builtins.so.13.0'
#             if not dst.exists(): dst.symlink_to(target)
#         except Exception as e:
#             print(f'[nvrtc-fix] Could not link into {sys_dir}: {e}')
# if shutil.which('ldconfig'):
#     subprocess.run(['ldconfig'], check=False)
# for name in ['libnvrtc.so', 'libnvrtc-builtins.so.13.0', 'libnvrtc-builtins.so.12']:
#     try: ctypes.CDLL(name); print(f'[nvrtc-fix] OK: {name}')
#     except OSError as e: print(f'[nvrtc-fix] MISSING: {name} -> {e}')
# print('[nvrtc-fix] Done. Restart runtime once, then rerun from Cell 2.')
print('[nvrtc-fix] Colab NVRTC fix cell skipped (not in Colab).')

In [ ]:
from model_utils import load_model_and_processor
model, processor = load_model_and_processor(cfg)

In [ ]:
from dataset import load_dataset_samples
samples = load_dataset_samples(cfg)
print(f'Loaded {len(samples)} samples.')

In [ ]:
import gc
import json
import time
import torch
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm as tqdm_nb

from model_utils import (
    generate_caption, get_tokenizer, get_image_token_positions,
    get_token_probabilities, get_content_token_mask, build_tf_inputs,
)
from saliency import get_saliency_fn
from evaluation import evaluate_faithfulness_per_token, evaluate_faithfulness_random
from visualization import (
    save_token_saliency_grid, save_comparison_figure, save_perturbation_curve,
)

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / 'config.json', 'w') as f:
    json.dump(vars(cfg), f, indent=2, default=str)

tok = get_tokenizer(processor)
all_sample_results = []
t_total = time.time()

for i in tqdm_nb(range(len(samples)), desc='Samples'):
    sample      = samples[i]
    image       = sample['image']
    ref_caption = sample.get('caption', '')
    sample_id   = sample.get('id', str(i))

    # 1. Generate caption
    gen_ids, gen_text, input_len, inputs = generate_caption(model, processor, image, cfg)
    total_len = gen_ids.shape[1]
    num_gen   = total_len - input_len
    print(f'\nSample {i}: {num_gen} tokens \u2014 {gen_text[:80]}')
    if num_gen == 0:
        continue

    # 2. Setup
    img_positions     = get_image_token_positions(inputs, cfg.image_token_index)
    tf_inputs         = build_tf_inputs(inputs, gen_ids, input_len)
    orig_probs        = get_token_probabilities(model, tf_inputs, gen_ids, input_len)
    token_strings     = {
        pos: tok.decode([gen_ids[0, pos].item()], skip_special_tokens=True).strip()
        for pos in range(input_len, total_len)
    }
    content_mask      = get_content_token_mask(tok, gen_ids, input_len)
    eval_content_mask = list(content_mask) if CONTENT_ONLY else None
    content_positions = [
        pos for pos, keep in zip(range(input_len, total_len), content_mask) if keep
    ]

    sample_dir = out_dir / f'sample_{i:04d}'
    if cfg.save_visualizations:
        sample_dir.mkdir(parents=True, exist_ok=True)

    # 3. Per-method saliency + deletion evaluation
    method_results = {}
    all_saliency   = {}
    for method in cfg.methods:
        sal_maps = get_saliency_fn(method)(model, tf_inputs, gen_ids, input_len, img_positions, cfg)
        all_saliency[method] = sal_maps
        ev = evaluate_faithfulness_per_token(
            model, inputs, gen_ids, input_len, sal_maps, orig_probs, cfg,
            content_mask=eval_content_mask,
        )
        for row in ev.get('per_token', []):
            row['token_text'] = token_strings.get(row.get('position'), '')
        print(f'  {method}: AOPC={ev["aopc"]:.4f}')
        method_results[method] = ev

        if cfg.save_visualizations:
            save_token_saliency_grid(
                image, sal_maps, token_strings, method,
                str(sample_dir / f'saliency_{method}.png'),
                content_positions=content_positions or None,
            )

    # 4. Random baseline
    random_positions = (
        [p for p, k in zip(range(input_len, total_len), content_mask) if k]
        if CONTENT_ONLY else list(range(input_len, total_len))
    )
    rand_ev = evaluate_faithfulness_random(
        model, inputs, gen_ids, input_len, orig_probs, cfg,
        content_mask=content_mask if CONTENT_ONLY else None,
        token_positions=random_positions,
    )
    for row in rand_ev.get('per_token', []):
        row['token_text'] = token_strings.get(row.get('position'), '')
    print(f'  random: AOPC={rand_ev["aopc"]:.4f}')
    method_results['random'] = rand_ev

    # 5. Visualizations
    if cfg.save_visualizations:
        if len(all_saliency) >= 2:
            save_comparison_figure(
                image, all_saliency, token_strings,
                str(sample_dir / 'comparison.png'),
                content_positions=content_positions or None,
            )
        save_perturbation_curve(
            method_results, str(sample_dir / 'perturbation_curve.png'),
            title=f'Sample {i}',
        )
        image.save(str(sample_dir / 'original.png'))

    all_sample_results.append({
        'sample_id':            sample_id,
        'sample_idx':           i,
        'generated_text':       gen_text,
        'reference_caption':    ref_caption,
        'num_generated_tokens': num_gen,
        'num_content_tokens':   sum(content_mask),
        'eval': {
            m: {
                'aopc':                r.get('aopc', 0.0),
                'mean_drops_by_ratio': r.get('mean_drops_by_ratio', {}),
                'per_token':           r.get('per_token', []),
            }
            for m, r in method_results.items()
        },
    })

    del tf_inputs, orig_probs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

elapsed = time.time() - t_total
print(f'\nDone: {len(all_sample_results)} samples in {elapsed:.0f}s')
print(f'Output: {out_dir}')

## Save Results
Flatten per-token rows and write `per_token_drops.csv` + `summary.csv`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

out_dir = Path(cfg.output_dir)

# ── Per-token drops CSV ───────────────────────────────────────────────
rows = []
for res in all_sample_results:
    base = {'sample_idx': res['sample_idx'], 'sample_id': res['sample_id']}
    for method, ev in res.get('eval', {}).items():
        for r in ev.get('per_token', []):
            rows.append({**base, 'method': method, **r})

csv_path = out_dir / 'per_token_drops.csv'
pd.DataFrame(rows).to_csv(csv_path, index=False)
print(f'Saved {len(rows)} rows -> {csv_path}')

# ── AOPC summary ──────────────────────────────────────────────────
all_eval = {}
for res in all_sample_results:
    for m, ev in res.get('eval', {}).items():
        all_eval.setdefault(m, []).append(ev.get('aopc', 0.0))

summary = pd.DataFrame([
    {'method': m, 'mean_aopc': np.mean(v), 'std_aopc': np.std(v), 'n_samples': len(v)}
    for m, v in all_eval.items()
]).sort_values('mean_aopc', ascending=False).reset_index(drop=True)

summary.to_csv(out_dir / 'summary.csv', index=False)
print(summary.to_string(index=False))